In [14]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        (os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [15]:
import os, cv2
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from sklearn.model_selection import train_test_split


In [16]:
image_dir = "/kaggle/input/x-ray-segmentation/X ray image/image"
mask_dir  = "/kaggle/input/x-ray-segmentation/X ray image/masks"

IMG_SIZE = 224


In [17]:
def load_data(image_dir, mask_dir, img_size=224):
    images, masks = [], []

    image_files = sorted(os.listdir(image_dir))
    mask_files  = sorted(os.listdir(mask_dir))

    for img_name, mask_name in zip(image_files, mask_files):
        img_path  = os.path.join(image_dir, img_name)
        mask_path = os.path.join(mask_dir, mask_name)

        img  = cv2.imread(img_path)
        mask = cv2.imread(mask_path, 0)

        if img is None or mask is None:
            print("❌ Missing:", img_path, mask_path)
            continue

        img  = cv2.resize(img, (img_size, img_size))
        mask = cv2.resize(mask, (img_size, img_size))

        img  = img / 255.0
        mask = mask / 255.0

        # 🔥 Force strict binary mask
        mask = np.where(mask > 0.5, 1.0, 0.0)

        images.append(img)
        masks.append(np.expand_dims(mask, axis=-1))

    return np.array(images), np.array(masks)

X, Y = load_data(image_dir, mask_dir, IMG_SIZE)
print("✅ Images:", X.shape)
print("✅ Masks :", Y.shape)

if len(X) == 0:
    raise ValueError("❌ Dataset EMPTY! Check your image/mask paths.")


✅ Images: (1867, 224, 224, 3)
✅ Masks : (1867, 224, 224, 1)


In [18]:
X_train, X_val, Y_train, Y_val = train_test_split(
    X, Y, test_size=0.2, random_state=42
)

print("✅ Train:", X_train.shape)
print("✅ Val  :", X_val.shape)


✅ Train: (1493, 224, 224, 3)
✅ Val  : (374, 224, 224, 3)


In [19]:
def dice_loss(y_true, y_pred, smooth=1):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    # Ensure shape: (B, H, W, 1)
    if len(y_true.shape) == 3:
        y_true = tf.expand_dims(y_true, axis=-1)
    if len(y_pred.shape) == 3:
        y_pred = tf.expand_dims(y_pred, axis=-1)

    intersection = tf.reduce_sum(y_true * y_pred)
    return 1 - (2. * intersection + smooth) / (
        tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth
    )


def weighted_bce_dice_loss(y_true, y_pred, pos_weight=5):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    # Ensure shape: (B, H, W, 1)
    if len(y_true.shape) == 3:
        y_true = tf.expand_dims(y_true, axis=-1)
    if len(y_pred.shape) == 3:
        y_pred = tf.expand_dims(y_pred, axis=-1)

    # Binary Crossentropy (per-pixel)
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)

    # Expand bce to (B,H,W,1)
    if len(bce.shape) == 3:
        bce = tf.expand_dims(bce, axis=-1)

    # Apply positive class weighting
    weight = y_true * pos_weight + (1 - y_true)
    bce = bce * weight

    # Dice loss
    dloss = dice_loss(y_true, y_pred)

    return tf.reduce_mean(bce) + dloss


In [20]:
# def dice_coef(y_true, y_pred, smooth=1):
#     y_true = tf.cast(y_true, tf.float32)
#     y_pred = tf.cast(y_pred, tf.float32)
#     intersection = tf.reduce_sum(y_true * y_pred)
#     return (2. * intersection + smooth) / (
#         tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth
#     )

# def iou(y_true, y_pred, smooth=1):
#     y_true = tf.cast(y_true, tf.float32)
#     y_pred = tf.cast(y_pred, tf.float32)
#     intersection = tf.reduce_sum(y_true * y_pred)
#     union = tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) - intersection
#     return (intersection + smooth) / (union + smooth)


In [21]:
# def dice_loss(y_true, y_pred, smooth=1):
#     y_true = tf.cast(y_true, tf.float32)
#     y_pred = tf.cast(y_pred, tf.float32)
#     intersection = tf.reduce_sum(y_true * y_pred)
#     return 1 - (2. * intersection + smooth) / (
#         tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth
#     )

# def weighted_bce_dice_loss(y_true, y_pred, pos_weight=5):
#     y_true = tf.cast(y_true, tf.float32)
#     y_pred = tf.cast(y_pred, tf.float32)

#     bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
#     weight = y_true * pos_weight + (1 - y_true)
#     bce = bce * weight

#     dloss = dice_loss(y_true, y_pred)
#     return tf.reduce_mean(bce) + dloss


In [22]:
def attention_gate(x, g, inter_channels):
    theta_x = layers.Conv2D(inter_channels, 1, padding="same")(x)
    phi_g   = layers.Conv2D(inter_channels, 1, padding="same")(g)
    add = layers.Add()([theta_x, phi_g])
    relu = layers.Activation("relu")(add)

    psi = layers.Conv2D(1, 1, padding="same")(relu)
    sigmoid = layers.Activation("sigmoid")(psi)

    return layers.Multiply()([x, sigmoid])


In [23]:
def build_hybrid_model(input_shape=(224,224,3)):
    inputs = layers.Input(input_shape)

    # 🔹 Encoder: Pretrained EfficientNet
    base_model = EfficientNetB0(
        include_top=False, weights="imagenet", input_tensor=inputs
    )

    c1 = base_model.get_layer("block2a_expand_activation").output   # 56x56
    c2 = base_model.get_layer("block3a_expand_activation").output   # 28x28
    c3 = base_model.get_layer("block4a_expand_activation").output   # 14x14
    c4 = base_model.get_layer("block6a_expand_activation").output   # 7x7

    # 🔹 Bottleneck
    b = layers.Conv2D(512, 3, activation="relu", padding="same")(c4)
    b = layers.BatchNormalization()(b)
    b = layers.Conv2D(512, 3, activation="relu", padding="same")(b)

    # 🔹 Decoder with Attention
    u1 = layers.UpSampling2D()(b)
    a1 = attention_gate(c3, u1, 256)
    u1 = layers.Concatenate()([u1, a1])
    u1 = layers.Conv2D(256, 3, activation="relu", padding="same")(u1)

    u2 = layers.UpSampling2D()(u1)
    a2 = attention_gate(c2, u2, 128)
    u2 = layers.Concatenate()([u2, a2])
    u2 = layers.Conv2D(128, 3, activation="relu", padding="same")(u2)

    u3 = layers.UpSampling2D()(u2)
    a3 = attention_gate(c1, u3, 64)
    u3 = layers.Concatenate()([u3, a3])
    u3 = layers.Conv2D(64, 3, activation="relu", padding="same")(u3)

    # 🔹 Output
    outputs = layers.Lambda(lambda x: tf.image.resize(x, (224,224)))(u3)
    outputs = layers.Conv2D(1, 1, activation="sigmoid")(outputs)

    model = models.Model(inputs, outputs)
    return model


In [24]:
model = build_hybrid_model()
model.summary()

# 🔒 Freeze EfficientNet for stable training
for layer in model.layers:
    if "efficientnet" in layer.name.lower():
        layer.trainable = False


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_2         │ (None, 224, 224,  │          0 │ input_layer_1[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization_1     │ (None, 224, 224,  │          7 │ rescaling_2[0][0] │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_3         │ (None, 224, 224,  │          0 │ normalization_1[… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 225, 225,  │          0 │ rescaling_3[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 112, 112,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 112, 112,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 112, 112,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 112, 112,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 112, 112,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 112, 112,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 112, 112,  │          0 │ block1a_activati… │
│ (Multiply)          │ 32)               │            │ block1a_se_expan… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_project_co… │ (None, 112, 112,  │        512 │ block1a_se_excit

 Total params: 8,987,211 (34.28 MB)

 Trainable params: 8,968,676 (34.21 MB)

 Non-trainable params: 18,535 (72.41 KB)

In [25]:
# model.compile(
#     optimizer=tf.keras.optimizers.Adam(1e-4),
#     loss=weighted_bce_dice_loss,
#     metrics=[dice_coef, iou]
# )


In [26]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=weighted_bce_dice_loss,
    metrics=[dice_coef, iou]
)


In [27]:
callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        "best_hybrid_model.keras", save_best_only=True,
        monitor="val_dice_coef", mode="max"
    ),
    tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(patience=5, factor=0.3)
]


In [ ]:
history_1 = model.fit(
    X_train, Y_train,
    validation_data=(X_val, Y_val),
    epochs=20,
    batch_size=8,
    callbacks=callbacks
)


Epoch 1/20


I0000 00:00:1768128836.988279     106 cuda_dnn.cc:529] Loaded cuDNN version 90300
E0000 00:00:1768128845.115587     106 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1768128845.300373     106 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


  1/187 ━━━━━━━━━━━━━━━━━━━━ 3:09:45 61s/step - dice_coef: 0.0506 - iou: 0.0260 - loss: 2.3242

I0000 00:00:1768128867.022085     106 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


186/187 ━━━━━━━━━━━━━━━━━━━━ 0s 84ms/step - dice_coef: 0.1719 - iou: 0.0987 - loss: 1.1584

E0000 00:00:1768128892.558934     107 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
E0000 00:00:1768128892.744186     107 gpu_timer.cc:82] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.


187/187 ━━━━━━━━━━━━━━━━━━━━ 122s 325ms/step - dice_coef: 0.1728 - iou: 0.0992 - loss: 1.1568 - val_dice_coef: 0.0019 - val_iou: 0.0010 - val_loss: 1.8782 - learning_rate: 1.0000e-04
Epoch 2/20
187/187 ━━━━━━━━━━━━━━━━━━━━ 18s 97ms/step - dice_coef: 0.4290 - iou: 0.2809 - loss: 0.7735 - val_dice_coef: 0.0487 - val_iou: 0.0255 - val_loss: 1.5673 - learning_rate: 1.0000e-04
Epoch 3/20
187/187 ━━━━━━━━━━━━━━━━━━━━ 18s 97ms/step - dice_coef: 0.5298 - iou: 0.3719 - loss: 0.6332 - val_dice_coef: 0.0635 - val_iou: 0.0336 - val_loss: 1.5597 - learning_rate: 1.0000e-04
Epoch 4/20
187/187 ━━━━━━━━━━━━━━━━━━━━ 18s 96ms/step - dice_coef: 0.6036 - iou: 0.4439 - loss: 0.5365 - val_dice_coef: 0.1173 - val_iou: 0.0660 - val_loss: 1.6482 - learning_rate: 1.0000e-04
Epoch 5/20
187/187 ━━━━━━━━━━━━━━━━━━━━ 17s 91ms/step - dice_coef: 0.6479 - iou: 0.4907 - loss: 0.4730 - val_dice_coef: 0.0489 - val_iou: 0.0261 - val_loss: 2.7045 - learning_rate: 1.0000e-04
Epoch 6/20
187/187 ━━━━━━━━━━━━━━━━━━━━ 17s 91ms/

In [ ]:
# 🔓 Unfreeze all layers for fine-tuning
for layer in model.layers:
    layer.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),  # lower LR for fine-tuning
    loss=weighted_bce_dice_loss,
    metrics=[dice_coef, iou]
)

history_2 = model.fit(
    X_train, Y_train,
    validation_data=(X_val, Y_val),
    epochs=20,
    batch_size=8,
    callbacks=callbacks
)
